# 06 — Human-Centered Explanation Generator

**Component 2 of the Explainable AI Decision Framework.**

Takes the raw SHAP output from `04_SHAPAnalysis.ipynb` (v2 model, 14 clean
features) and converts it into language a stressed 19-year-old could read and
understand in one pass.

**Hard rule** (`.claude/skills/explainable-ai/SKILL.md`): the student never
sees a SHAP value, a feature name, a numeric weight, or any ML terminology.
Everything technical in this notebook is for our verification only.

The generation logic lives in `ml_pipeline/src/explainability/` as reusable,
testable code — this notebook only exercises it against the same four local
cases examined in `04_SHAPAnalysis.ipynb`.

In [1]:
import json
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import joblib
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import train_test_split

from src.explainability import generate_explanation, severity_contributions
from src.explainability.generator import save_faithfulness_log
from src.explainability.templates import FEATURE_PHRASES

ARTIFACTS_DIR = Path("../artifacts")
DATA_PATH = "../datasets/raw/student_stress_factors.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.2

print(f"Plain-language templates defined for {len(FEATURE_PHRASES)} features, "
      f"both directions each.")

Plain-language templates defined for 14 features, both directions each.


## Load the v2 model and reproduce the same SHAP setup as notebook 04

In [2]:
model = joblib.load(ARTIFACTS_DIR / "stress_model_v2.pkl")
with open(ARTIFACTS_DIR / "shap_config.json", encoding="utf-8") as f:
    shap_config = json.load(f)

FEATURES = shap_config["feature_order"]
CLASS_LABELS = shap_config["target"]["classes"]
CLASS_MEANING = shap_config["target"]["class_meaning"]
TARGET = shap_config["target"]["name"]
HIGH_STRESS_IDX = CLASS_LABELS.index(2)

df = pd.read_csv(DATA_PATH)
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print(f"v2 model, {len(FEATURES)} features; SHAP array {shap_values.shape}")

v2 model, 14 features; SHAP array (220, 14, 3)


## Select the same four cases as `04_SHAPAnalysis.ipynb`

Three confidently-correct predictions (one per class) plus the most confident
misclassification — the last is the one that matters most for deployment
risk.

In [3]:
confidence = y_proba.max(axis=1)
correct = y_pred == y_test.values

selected = {}
for c in CLASS_LABELS:
    mask = correct & (y_test.values == c)
    if mask.any():
        selected[f"correct_class_{c}_{CLASS_MEANING[str(c)]}"] = int(
            np.where(mask)[0][np.argmax(confidence[mask])]
        )

wrong = np.where(~correct)[0]
if len(wrong):
    selected[
        f"MISCLASSIFIED_true{y_test.values[wrong[np.argmax(confidence[wrong])]]}"
        f"_pred{y_pred[wrong[np.argmax(confidence[wrong])]]}"
    ] = int(wrong[np.argmax(confidence[wrong])])

for name, idx in selected.items():
    print(f"{name:34s} row {idx:3d}  true={y_test.values[idx]}  "
          f"pred={y_pred[idx]}  confidence={confidence[idx]:.3f}")

correct_class_0_low                row   0  true=0  pred=0  confidence=1.000
correct_class_1_moderate           row   8  true=1  pred=1  confidence=1.000
correct_class_2_high               row   2  true=2  pred=2  confidence=1.000
MISCLASSIFIED_true1_pred2          row  29  true=1  pred=2  confidence=0.602


## Which SHAP axis should drive the explanation?

`shap_values` is `(n_samples, n_features, n_classes)`, so the explanation must
choose *which* class column to read. This is not a cosmetic choice — it
determines whether "raising" and "easing" mean anything.

Three candidates, evaluated on the four cases below:

- **(a) High-stress column only.** Describes distance from the top of the
  scale. For a moderate prediction nearly every contribution is negative
  (pushing away from high), so the explanation degenerates into an
  all-protective list that contradicts the stated moderate level.
- **(b) The predicted class's own column.** Its sign means "pushed toward this
  outcome", not "increased stress". For a low-stress prediction every strong
  contributor is positive, which would report met basic needs and good academic
  performance as *raising* stress — semantically inverted.
- **(c) Severity axis, `SHAP(high) - SHAP(low)`.** Measures movement toward the
  severe end relative to the calm end. Respects the target's ordering.

The cell below is the evidence for choosing (c).

In [4]:
axis_rows = []
for name, idx in selected.items():
    p = int(y_pred[idx])
    axes = {
        "(a) high column": shap_values[idx, :, HIGH_STRESS_IDX],
        "(b) predicted class": shap_values[idx, :, CLASS_LABELS.index(p)],
        "(c) severity": severity_contributions(shap_values[idx], CLASS_LABELS),
    }
    for label, arr in axes.items():
        top = np.argsort(np.abs(arr))[::-1][:4]
        axis_rows.append({
            "case": name[:26],
            "pred": p,
            "axis": label,
            "raising_of_top4": int(sum(arr[i] > 0 for i in top)),
            "top_factor": FEATURES[top[0]],
        })

axis_cmp = pd.DataFrame(axis_rows)
print(axis_cmp.pivot(index=["case", "pred"], columns="axis",
                     values="raising_of_top4").to_string())
print("\nCounts are 'how many of the top 4 factors are labelled raising'.")
print("Coherent = 0/4 for a low prediction, a genuine mix for moderate, 4/4 for high.")

axis                            (a) high column  (b) predicted class  (c) severity
case                      pred                                                    
MISCLASSIFIED_true1_pred2 2                   3                    3             3
correct_class_0_low       0                   0                    4             0
correct_class_1_moderate  1                   0                    4             2
correct_class_2_high      2                   4                    4             4

Counts are 'how many of the top 4 factors are labelled raising'.
Coherent = 0/4 for a low prediction, a genuine mix for moderate, 4/4 for high.


## Side by side: technical attribution vs. what the student actually sees

The left-hand block is what SHAP produced. The right-hand block is the only
thing that would ever reach a student.

In [5]:
explanations = {}

for name, idx in selected.items():
    # Signed severity axis, not a single class column - see
    # severity_contributions() and the axis comparison below.
    contributions = severity_contributions(shap_values[idx], CLASS_LABELS)
    predicted_class = int(y_pred[idx])

    explanation = generate_explanation(
        shap_values=contributions,
        feature_values=X_test.iloc[idx].values,
        feature_names=FEATURES,
        predicted_class=predicted_class,
        top_n=4,
        context={
            "case": name,
            "test_row": idx,
            "true_class": int(y_test.values[idx]),
            "predicted_class": predicted_class,
            "model_confidence": float(confidence[idx]),
            "correct": bool(correct[idx]),
            "model_version": shap_config["model_version"],
        },
    )
    explanations[name] = explanation

    print("=" * 78)
    print(f"CASE: {name}   (test row {idx})")
    print(f"true={y_test.values[idx]} ({CLASS_MEANING[str(y_test.values[idx])]})  "
          f"pred={predicted_class} ({CLASS_MEANING[str(predicted_class)]})  "
          f"confidence={confidence[idx]:.3f}")
    print("=" * 78)

    print("\n--- TECHNICAL (internal only - never shown to a student) ---")
    tech = pd.DataFrame(
        {
            "feature_value": [f.feature_value for f in explanation.factors],
            "severity_contribution": [f.shap_value for f in explanation.factors],
            "direction": [f.direction for f in explanation.factors],
        },
        index=[f.feature for f in explanation.factors],
    )
    print(tech.round(4).to_string())
    print(f"coverage of total |SHAP|: {explanation.faithfulness_record['coverage_ratio']:.1%}")

    print("\n--- PLAIN LANGUAGE (what the student sees) ---")
    print(explanation.user_facing())
    print()

CASE: correct_class_0_low   (test row 0)
true=0 (low)  pred=0 (low)  confidence=1.000

--- TECHNICAL (internal only - never shown to a student) ---
                              feature_value  severity_contribution direction
academic_performance                    5.0                -0.1618    easing
basic_needs                             4.0                -0.1601    easing
teacher_student_relationship            5.0                -0.1504    easing
headache                                1.0                -0.1112    easing
coverage of total |SHAP|: 58.8%

--- PLAIN LANGUAGE (what the student sees) ---
Your current wellbeing check-in suggests things feel reasonably steady for you at the moment. What does seem to be steadying things is that how your studies have been going seems to be working in your favour. It also helps that having your everyday essentials reliably met seems to be giving you a solid foundation, and feeling supported by teaching staff seems to be helping you. This i

## Deployment-risk check on the misclassified case

The question is not "is the explanation correct" — by construction it
faithfully describes what the model did. The question is whether a student who
received it, **given the prediction was wrong**, would be harmed, misled, or
pushed away from help they actually need.

In [6]:
mis_key = next(k for k in explanations if k.startswith("MISCLASSIFIED"))
mis = explanations[mis_key]
rec = mis.faithfulness_record["context"]

print(f"Case: {mis_key}")
print(f"True class {rec['true_class']} ({CLASS_MEANING[str(rec['true_class'])]}), "
      f"model said {rec['predicted_class']} ({CLASS_MEANING[str(rec['predicted_class'])]})")
print(f"Model confidence: {rec['model_confidence']:.3f}\n")

print("Text the student would have received:")
print("-" * 70)
print(mis.user_facing())
print("-" * 70)

checks = {
    "Contains no diagnostic or clinical claim": True,  # enforced by validate_user_facing_text
    "Frames result as a changeable snapshot, not a verdict": "snapshot" in mis.paragraph,
    "Avoids telling the student they are fine / dismissing concern":
        "no problem" not in mis.paragraph.lower() and "nothing to worry" not in mis.paragraph.lower(),
    "Describes factors the student can recognise in their own life": len(mis.factors) >= 3,
    "Understates rather than overstates certainty ('appears', 'seems')":
        "appears" in mis.paragraph or "seems" in mis.paragraph or "suggests" in mis.paragraph,
}
print("\nDeployment-risk checks:")
for label, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {label}")

Case: MISCLASSIFIED_true1_pred2
True class 1 (moderate), model said 2 (high)
Model confidence: 0.602

Text the student would have received:
----------------------------------------------------------------------
Your current wellbeing check-in suggests you are under a fair amount of pressure at the moment. In particular, how you have been feeling about yourself lately appears to be weighing on you. Alongside that, how your studies have been going appears to be weighing on you, and you may be missing consistent support from the people around you. On the other side, being able to breathe comfortably and settle physically seems to be one of the things going well for you. This is a snapshot of how things look right now rather than a fixed picture. If this feels familiar or has been going on for a while, talking it through with your university wellbeing service is a reasonable next step.
----------------------------------------------------------------------

Deployment-risk checks:
  [PASS] 

## Faithfulness log

Every generated explanation is logged with the SHAP attribution it was derived
from — feature, raw value, signed contribution, chosen direction, and the
resulting phrase — so the plain-language text can later be audited against the
attribution it claims to describe. This is the "faithfulness" criterion in
`.claude/skills/explainable-ai/SKILL.md`, and it is what makes the Phase 8
explanation-quality evaluation possible.

This log is never surfaced to a student.

In [7]:
log_path = save_faithfulness_log(
    list(explanations.values()),
    Path("../experiments") / "faithfulness_log_v2_local_cases.jsonl",
)
print(f"Wrote {log_path}\n")

sample = explanations[list(explanations)[0]].faithfulness_record
print("Example record (first case), truncated:")
print(json.dumps({k: v for k, v in sample.items() if k != "paragraph"}, indent=2)[:1200])

Wrote ../experiments/faithfulness_log_v2_local_cases.jsonl

Example record (first case), truncated:
{
  "generated_utc": "2026-08-15T17:24:50Z",
  "predicted_class": 0,
  "top_n": 4,
  "factors": [
    {
      "feature": "academic_performance",
      "feature_value": 5.0,
      "shap_value": -0.16183124023183765,
      "direction": "easing",
      "rank": 1,
      "phrase": "how your studies have been going seems to be working in your favour"
    },
    {
      "feature": "basic_needs",
      "feature_value": 4.0,
      "shap_value": -0.16005703777562627,
      "direction": "easing",
      "rank": 2,
      "phrase": "having your everyday essentials reliably met seems to be giving you a solid foundation"
    },
    {
      "feature": "teacher_student_relationship",
      "feature_value": 5.0,
      "shap_value": -0.1503841988645892,
      "direction": "easing",
      "rank": 3,
      "phrase": "feeling supported by teaching staff seems to be helping you"
    },
    {
      "feature": "h

In [8]:
# Faithfulness spot-check: does every phrase's stated direction match the sign
# of the SHAP value it came from? This is the check the log exists to enable.
rows = []
for name, exp in explanations.items():
    for f in exp.factors:
        rows.append({
            "case": name[:28],
            "feature": f.feature,
            "shap": round(f.shap_value, 4),
            "stated_direction": f.direction,
            "sign_matches": (f.shap_value > 0) == (f.direction == "raising"),
        })
check = pd.DataFrame(rows)
print(check.to_string(index=False))
print(f"\nDirection/sign mismatches: {(~check['sign_matches']).sum()} of {len(check)}")

                     case                      feature    shap stated_direction  sign_matches
      correct_class_0_low         academic_performance -0.1618           easing          True
      correct_class_0_low                  basic_needs -0.1601           easing          True
      correct_class_0_low teacher_student_relationship -0.1504           easing          True
      correct_class_0_low                     headache -0.1112           easing          True
 correct_class_1_moderate   extracurricular_activities -0.0809           easing          True
 correct_class_1_moderate                peer_pressure -0.0650           easing          True
 correct_class_1_moderate teacher_student_relationship  0.0570          raising          True
 correct_class_1_moderate                  basic_needs  0.0523          raising          True
     correct_class_2_high   extracurricular_activities  0.1437          raising          True
     correct_class_2_high                peer_pressure  0.14